# Analisis de entrenamiento Tiny ImageNet

Notebook generado automaticamente.

Incluye:
- carga de historial y resumen
- metricas principales
- mejor epoch
- curvas de cost, accuracy, tiempo y learning rate
- tabla resumen final


In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

csv_file = Path(r"runs\run_20260421_005806\history.csv")
json_file = Path(r"runs\run_20260421_005806\summary.json")

df = pd.read_csv(csv_file)
with open(json_file, 'r', encoding='utf-8') as f:
    summary = json.load(f)

df = df.sort_values('epoch').reset_index(drop=True)

if 'lr' not in df.columns:
    df['lr'] = None

df['generalization_gap'] = df['train'] - df['test']
df['test_gain'] = df['test'].diff().fillna(0)
df['cost_delta'] = df['cost'].diff().fillna(0)

df.head()


## Resumen del experimento

- Arquitectura: resnet18
- Pretrained: True
- Freeze backbone: False
- Optimizer: adamw
- Epochs completadas: 20
- Workers: 1
- Local epochs: 1
- LR inicial: 5e-05
- LR step size: 5
- LR gamma: 0.5
- Batch size train: 128
- Batch size val: 256
- Image size: 128
- Device: cpu
- Mejor test: 0.6863
- Mejor epoch: 16
- Tiempo total: 13000.078450918198 s


In [ ]:
print('Columnas disponibles:')
print(df.columns.tolist())
print()
print('Primeras filas:')
display(df.head())
print()
print('Ultimas filas:')
display(df.tail())


In [ ]:
best_idx = df['test'].idxmax()
best_row = df.loc[best_idx]
last_row = df.iloc[-1]

print('=== MEJOR EPOCH ===')
print(f"Epoch: {int(best_row['epoch'])}")
print(f"Cost: {best_row['cost']:.6f}")
print(f"Train: {best_row['train']:.6f}")
print(f"Test: {best_row['test']:.6f}")
print(f"Gap train-test: {best_row['generalization_gap']:.6f}")
if pd.notna(best_row['lr']):
    print(f"LR: {best_row['lr']:.8f}")
print()
print('=== ULTIMA EPOCH ===')
print(f"Epoch: {int(last_row['epoch'])}")
print(f"Cost: {last_row['cost']:.6f}")
print(f"Train: {last_row['train']:.6f}")
print(f"Test: {last_row['test']:.6f}")
print(f"Gap train-test: {last_row['generalization_gap']:.6f}")
print(f"Epoch time: {last_row['epoch_time']:.2f} s")
print(f"Total time: {last_row['total_time']:.2f} s")


In [ ]:
summary_table = pd.DataFrame([
    {
        'metric': 'best_test',
        'value': float(df['test'].max())
    },
    {
        'metric': 'best_epoch',
        'value': int(df.loc[df['test'].idxmax(), 'epoch'])
    },
    {
        'metric': 'final_test',
        'value': float(df.iloc[-1]['test'])
    },
    {
        'metric': 'final_train',
        'value': float(df.iloc[-1]['train'])
    },
    {
        'metric': 'final_cost',
        'value': float(df.iloc[-1]['cost'])
    },
    {
        'metric': 'total_time_sec',
        'value': float(df.iloc[-1]['total_time'])
    },
])

summary_table


## Graficas principales


In [ ]:
plt.figure(figsize=(9,5))
plt.plot(df['epoch'], df['cost'], marker='o', label='Cost')
plt.axvline(df.loc[df['test'].idxmax(), 'epoch'], linestyle='--', label='Best epoch')
plt.title('Cost por epoch')
plt.xlabel('Epoch')
plt.ylabel('Cost')
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(9,5))
plt.plot(df['epoch'], df['train'], marker='o', label='Train accuracy')
plt.plot(df['epoch'], df['test'], marker='s', label='Test accuracy')
plt.axvline(df.loc[df['test'].idxmax(), 'epoch'], linestyle='--', label='Best epoch')
plt.title('Accuracy por epoch')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(9,5))
plt.plot(df['epoch'], df['generalization_gap'], marker='o')
plt.axhline(0, linestyle='--')
plt.title('Gap de generalizacion (train - test)')
plt.xlabel('Epoch')
plt.ylabel('Gap')
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(9,5))
plt.plot(df['epoch'], df['epoch_time'], marker='o', label='Epoch time')
plt.plot(df['epoch'], df['total_time'], marker='s', label='Total time')
plt.title('Tiempo de entrenamiento')
plt.xlabel('Epoch')
plt.ylabel('Segundos')
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
if df['lr'].notna().any():
    plt.figure(figsize=(9,5))
    plt.plot(df['epoch'], df['lr'], marker='o')
    plt.title('Learning Rate por epoch')
    plt.xlabel('Epoch')
    plt.ylabel('LR')
    plt.grid(True)
    plt.tight_layout()
    plt.show()
else:
    print('No hay columna lr disponible para graficar.')


## Analisis de progreso


In [ ]:
progress_table = pd.DataFrame([
    {
        'metric': 'delta_train',
        'value': float(df.iloc[-1]['train'] - df.iloc[0]['train'])
    },
    {
        'metric': 'delta_test',
        'value': float(df.iloc[-1]['test'] - df.iloc[0]['test'])
    },
    {
        'metric': 'delta_cost',
        'value': float(df.iloc[-1]['cost'] - df.iloc[0]['cost'])
    },
    {
        'metric': 'best_minus_final_test',
        'value': float(df['test'].max() - df.iloc[-1]['test'])
    },
])

progress_table


In [ ]:
print('Top 10 epochs por test accuracy:')
display(df.sort_values('test', ascending=False).head(10))


In [ ]:
print('Epochs con mayor ganancia en test:')
display(df.sort_values('test_gain', ascending=False).head(10))


## Conclusiones automaticas


In [ ]:
best_test = float(df['test'].max())
final_test = float(df.iloc[-1]['test'])
best_epoch = int(df.loc[df['test'].idxmax(), 'epoch'])
final_gap = float(df.iloc[-1]['generalization_gap'])
cost_trend = float(df.iloc[-1]['cost'] - df.iloc[0]['cost'])

print(f'Mejor test accuracy: {best_test:.6f} en epoch {best_epoch}')
print(f'Test final: {final_test:.6f}')
print(f'Gap final train-test: {final_gap:.6f}')
print(f'Cambio total en cost: {cost_trend:.6f}')

if final_test < best_test:
    print('Observacion: el mejor rendimiento ocurrio antes de la ultima epoch.')

if final_gap > 0.15:
    print('Observacion: puede haber sobreajuste porque train esta bastante por encima de test.')

if cost_trend < 0:
    print('Observacion: el cost total bajo respecto al inicio, lo cual es una buena senal.')
